# Throwing and Exception Contracts

In this lesson, you will learn to make methods report invalid input with exceptions and preserve the information callers need to understand a failure.

CSC-239 · Module 7 · Lesson 2 of 4

You have traced exceptions raised by Java operations. Now you will decide when your own methods should reject a value. A quantity-entry example will help you separate text that cannot become an integer from an integer that breaks an application rule.

Use the [Module 7 glossary](terms.md) to revisit the vocabulary after its explanation.

## Learning Goals

- Define, raise, and handle an application-specific checked exception, explaining the separate jobs of `throw` and `throws`.
- Convert text to an integer, enforce a stated value rule, and preserve an earlier parsing failure as the cause when one exists.

## Why This Matters

A method's useful behavior includes both its successful results and its response to invalid input. Returning a made-up number after a failed conversion can let later calculations continue with a value that was never valid. A clear failure contract lets the caller choose a response while keeping the failed operation from pretending to succeed.

Applications also need different levels of explanation. A person entering an order needs a useful message about the quantity. A developer investigating a problem may need the original conversion diagnostic. Keeping both supports a clear interface and makes faults easier to investigate.

This extends the method contracts and exception handlers you already know. Later, file operations and automated tests will use the same distinction between an expected result and a reported failure.

## Check Your Starting Point

Predict every output line without running the snippet. Identify which attempted print cannot finish, which later success message is skipped, and where execution continues after the handler.

```java
class StartingCheck {
    public static int share(int items, int groups) {
        return items / groups;
    }
}
try {
    System.out.println("Share: " + StartingCheck.share(12, 0));
    System.out.println("Calculated");
} catch (ArithmeticException problem) {
    System.out.println("Request failed");
}
System.out.println("Next request");
```

In [ ]:
My predicted output:

Unfinished print and skipped success message:

Where execution continues after the handler:


<details><summary>Show answer: follow the method contract</summary>

```text
Request failed
Next request
```

Division by zero interrupts the method call before it returns. The caller cannot finish its Share message and skips Calculated. The handler prints Request failed, then execution reaches Next request. The failed division is not retried.

</details>

## Video Demonstration

Follow conversion first and the range check second. Predict the successful value, each application message, and whether an original parsing exception is attached.

<video controls preload="metadata" width="960">
  <source src="media/02_throwing_and_exception_contracts/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/02_throwing_and_exception_contracts/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the throwing and exception contracts demonstration transcript](media/02_throwing_and_exception_contracts/transcript.md).


## Concept

### Give an invalid request a clear outcome

A campus supply assistant enters the number of notebooks requested for a workshop. Each entry arrives as text. The calculator accepts a decimal whole number that is zero or greater; zero means no notebooks are requested. It reports an accepted quantity or explains why that entry was rejected. It checks each entry independently so one bad entry does not discard later requests.

There are two checks. **Parsing** converts text into a value with a known type. **Validation** checks whether that value follows the application's rules. The text `"two"` cannot be parsed as an `int`. The text `"-1"` can be parsed, but a negative notebook quantity breaks this application's rule. Those failures need different messages.

Before combining the checks, we will build the smaller pieces: raise a failure, give that failure a useful type, and state what a caller must handle.

### Raise a failure where the rule is broken

The Java keyword **`throw`** raises a particular exception object immediately. It does not merely print a warning. Consider a separate rule that requires at least one group:

```java
class GroupRules {
    public static int requirePositive(int groups) {
        if (groups < 1) {
            throw new IllegalArgumentException("Groups must be positive.");
        }
        return groups;
    }
}
```

The `if` condition selects invalid counts. `new IllegalArgumentException(...)` constructs an exception object whose message describes the broken rule. **`IllegalArgumentException`** is a class name, not a keyword; it represents an inappropriate argument supplied to a method. Executing `throw` leaves the method before its ordinary `return` statement.

If `groups` is `3`, the condition is false and the method returns `3`. If it is `0`, the method raises the exception instead of returning a count. A caller can use the handler pattern from the previous lesson:

```java
try {
    System.out.println(GroupRules.requirePositive(0));
} catch (IllegalArgumentException problem) {
    System.out.println(problem.getMessage());
}
```

The output is `Groups must be positive.` The attempted result print cannot finish because its method call did not return a value. The handler prints the message carried by the exception object. This keeps the rule inside the method and the response in the caller.

### Understand what the compiler requires

A **checked exception** creates a catch-or-declare requirement in ordinary Java method code. If a method can let such an exception escape, its code must either handle it or declare the possible failure in its method header. Exceptions outside the `RuntimeException` and `Error` class families are checked.

An **unchecked exception** belongs to the `RuntimeException` or `Error` family. Java does not impose that catch-or-declare requirement for these types. `ArithmeticException`, which you handled earlier, and `IllegalArgumentException` are unchecked exceptions in the `RuntimeException` family.

These categories describe compiler rules, not how serious a failure is. A checked exception is not automatically harmless, and an unchecked exception is not automatically safe to ignore. Choose the handling behavior from the operation's requirements and what the caller can reasonably do next. The examples use specific expected types; broadly catching `Error` is not a routine recovery strategy.

### Name an application-specific failure

A **custom exception class** gives a particular kind of application failure its own type. For a checked count error, extend `Exception`:

```java
class InvalidCountException extends Exception {
    public InvalidCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
```

The `extends` relationship makes an `InvalidCountException` an `Exception`. Because this class is outside the unchecked families, it is a checked type. The constructor receives a message and a possible earlier failure. Its `super(message, cause)` call passes both to the superclass constructor, which provides the behavior used by `getMessage()` and `getCause()`.

**`Throwable`** is the common superclass of Java exception and error objects. Here it allows the constructor parameter to refer to different kinds of earlier failures. It is a type name, not a keyword. A **null reference**, written with the literal `null`, refers to no object. Passing `null` as the cause means that we are not attaching an earlier exception.

The new type does not print or handle anything by itself. It gives a caller a specific kind of failure to catch and carries the information that caller will inspect.

### Declare a failure that may reach the caller

The Java keyword **`throws`** introduces a method's declaration of possible escaping exception types. Put it after the parameter list:

```java
class CountRules {
    public static int read(int count) throws InvalidCountException {
        if (count < 0) {
            throw new InvalidCountException("Count cannot be negative.", null);
        }
        return count;
    }
}
```

The declaration tells a caller that `read` may fail with `InvalidCountException`. The executed `throw` inside the condition creates that failed path. For a nonnegative count, the branch is skipped and `return count` still works normally. A `throws` declaration does not force every call to fail.

A caller can meet the checked-exception requirement by handling it:

```java
try {
    System.out.println(CountRules.read(-2));
} catch (InvalidCountException problem) {
    System.out.println(problem.getMessage());
}
```

This prints `Count cannot be negative.` A helper method could instead declare `throws InvalidCountException` and let its own caller handle the failure. Adding a declaration does not repair the invalid input or resume a failed operation; it makes the helper's possible failure explicit.

IJava may accept some top-level checked calls that an ordinary Java method cannot leave undeclared. Keep the explicit handlers and method declarations shown here so the same failure contract works in conventional Java source files. Notebook acceptance alone is not evidence that omitting a required declaration is portable.

<details class="animation-panel" open>
<summary>A declaration and an executed throw — show or hide animation</summary>
<p><img src="media/02_throwing_and_exception_contracts/throw_and_throws.gif" alt="The declaration describes a failure that may escape; no exception has been raised. The branch containing throw is not executed. The application rule selects the failure branch. The method leaves without returning a count. The caller receives the object raised by the executed statement." width="960" style="max-width:100%;height:auto;"></p>
</details>

The method header declares a possible failure before any call is made. The acceptable count reaches its return statement. The negative count instead selects the branch that executes throw, so the caller receives an exception rather than a returned count. The declaration is the same on both paths; the supplied value determines which path runs.

This animation loops about every 13 seconds. [View the static diagram: a declaration and an executed throw](media/02_throwing_and_exception_contracts/throw_and_throws_still.png).

### Convert text before applying the quantity rule

`Integer.parseInt(text)` converts decimal integer text to an `int`. The result for `"3"` is `3`; the result for `"-1"` is `-1`. A minus sign can be part of a valid integer, even when the application will later reject that value.

The method raises **`NumberFormatException`** when it cannot perform the conversion. Words such as `"two"`, surrounding spaces such as `" 3 "`, and values outside the `int` range fail this conversion. `NumberFormatException` is an unchecked exception. Its class belongs to the `RuntimeException` family through `IllegalArgumentException`.

Parsing success answers only whether the text represents a supported integer. The quantity rule is a separate question. After obtaining an integer, check whether it is less than zero. That order lets the program explain which requirement failed.

### Keep the original failure when adding context

An **exception cause** is an earlier failure attached to another exception. It lets a method report an application-specific problem while retaining the diagnostic information that explains what went wrong underneath.

```java
try {
    return Integer.parseInt(text);
} catch (NumberFormatException cause) {
    throw new InvalidCountException("Count must be a whole number.", cause);
}
```

This fragment belongs inside a method that declares `throws InvalidCountException`. The variable `cause` refers to the original parsing exception caught by the handler. Passing that same reference to the new exception preserves the earlier object. The caller receives an `InvalidCountException` with a clearer count-related message, and the original `NumberFormatException` is still available through `getCause()`.

Compare that with a negative value that parsed successfully. The application can raise its own exception for the value rule and pass `null` as its cause. There was no earlier conversion exception to retain. A missing cause therefore does not mean that no failure occurred; it means this exception has no attached earlier failure.

When inspecting a cause, store and check the returned reference before calling a method through it:

```java
Throwable cause = problem.getCause();
if (cause != null) {
    System.out.println("Cause: " + cause.getMessage());
} else {
    System.out.println("Cause: none");
}
```

Here `problem` is the application exception already received by a handler. `!= null` asks whether a cause object is present. Calling `getMessage()` through a null reference would itself fail. The `else` branch lets a caller describe a valid range-only failure without inventing a parsing cause.

<details class="animation-panel" open>
<summary>Preserving an exception cause — show or hide animation</summary>
<p><img src="media/02_throwing_and_exception_contracts/cause_object_link.gif" alt="A NumberFormatException object records the parsing failure. The new object has its own message and retains the earlier object. The original diagnostic remains available to the caller. Construct the application exception with null cause; do not invent a parsing failure." width="960" style="max-width:100%;height:auto;"></p>
</details>

The application exception and the original parsing exception are separate objects. Passing the original object to the constructor preserves a reference to it, so the caller can inspect both the application message and the earlier diagnostic. A range-only failure has no earlier conversion exception, so its cause reference is null.

This animation loops about every 10.5 seconds. [View the static diagram: preserving an exception cause](media/02_throwing_and_exception_contracts/cause_object_link_still.png).

## Worked Example

### Step 1: give quantity failures a shared application type

The supply assistant needs one kind of quantity failure to handle, even though entries can fail for different reasons. `InvalidQuantityException` extends `Exception`, and its constructor keeps a message and a possible cause. This follows the custom-count pattern above with a name that fits the task.

### Step 2: separate conversion from the range guard

`QuantityParser.read` declares `throws InvalidQuantityException`. It first attempts `Integer.parseInt(text)` and stores the result in `quantity`. If conversion fails, its handler raises a new application exception with the whole-number message and the original parsing cause. That thrown failure leaves the method, so no range check or successful return follows it.

If conversion succeeds, execution reaches `if (quantity < 0)`. A negative value raises an application exception with the negative-quantity message and no earlier cause. Zero and positive values reach `return quantity`. The method therefore returns only values that meet both requirements.

### Step 3: respond to each independent entry

The caller processes `"3"`, `"two"`, and `"-1"` in that order. Its handler is inside the loop, so each request has its own success or failure response. A successful call prints `Quantity: ` and the returned number. A failed call prints `Problem: ` and the application message, followed by whether an earlier cause is present.

Read and run the complete program below. Connect each response with conversion first and validation second.

In [ ]:
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"3", "two", "-1"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}


The program produces:

```text
Quantity: 3
Problem: Quantity must be a whole number.
Has cause: true
Problem: Quantity cannot be negative.
Has cause: false
```

The first entry converts to `3` and meets the nonnegative rule, so its call returns a quantity. The word `"two"` fails during conversion. The application exception keeps the original parsing exception, which makes its cause check `true`.

The final entry converts to `-1` before the range guard rejects it. Its cause check is `false` because conversion did not fail. Both rejected entries still produce an application exception. The flag distinguishes whether an earlier failure is attached; it does not classify the entry as successful or unsuccessful.

You now have a complete path from input text through the two checks to the caller's response. The practice changes inputs and handler details so you can apply that reasoning to new cases. Other tasks may use a different value rule, so carry over the structure while checking the new contract.

<details class="animation-panel" open>
<summary>Conversion and the quantity rule — show or hide animation</summary>
<p><img src="media/02_throwing_and_exception_contracts/conversion_and_validation.gif" alt="Conversion and the application range rule are separate checks. Return 3; the caller prints Quantity: 3. Conversion fails before the range guard; wrap the original cause. The custom exception retains the original parsing exception. Conversion succeeds, but the application rule rejects the value. There is no earlier conversion exception to attach." width="960" style="max-width:100%;height:auto;"></p>
</details>

Follow each text entry through conversion first and the quantity rule second. The word cannot become an integer, so its range check is never reached. The negative value does become an integer, but the application rejects it afterward. Both failures receive a useful message; only the conversion failure has an earlier exception to retain.

This animation loops about every 15.5 seconds. [View the static diagram: conversion and the quantity rule](media/02_throwing_and_exception_contracts/conversion_and_validation_still.png).

## Guided Practice

### Separate conversion from a value rule

The supply assistant enters requested notebook quantities as text. The next program checks `"8"`, `"oops"`, and `"-3"` independently. It accepts quantities of zero or more, reports a useful message for each rejected entry, and reports whether the application exception retains an earlier cause.

Before running, predict every output line. For each rejected input, identify the operation or `throw` statement that starts its failed path. Explain why declaring `throws InvalidQuantityException` does not force the valid input to fail.

In [ ]:
Predicted complete output:

First failure and executed throw for each rejected input:

Why the declaration does not force a valid call to fail:


In [ ]:
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}


Run the program and record its complete output. Compare it with your prediction. Identify the first difference, if any, and explain the conversion or validation decision responsible. Explain how an application exception can have `Has cause: false` even though a failure occurred.

In [ ]:
Actual complete output:

First difference and explanation, or why the prediction matched:

Why an application failure can have no earlier cause:


Trace the same three inputs through the parser. For each input, state whether parsing succeeds, whether execution reaches the nonnegative check, which exception the caller receives if the call fails, and what earlier cause is attached.

Classify `NumberFormatException` and `InvalidQuantityException` as checked or unchecked using their class families. State what an ordinary Java method must do if it can let the checked type escape. Explain why these categories describe compiler obligations rather than the seriousness of a failure.

In [ ]:
Input "8": parsing / range check reached / caller result or exception / cause:

Input "oops": parsing / range check reached / caller result or exception / cause:

Input "-3": parsing / range check reached / caller result or exception / cause:

Exception categories and the class-family evidence:

Catch-or-declare obligation for an ordinary Java method:

Why checked and unchecked do not measure seriousness:


<details>
<summary>Show answer</summary>

The text "8" converts to 8 and passes the nonnegative rule, so Quantity: 8 prints. "oops" cannot convert: Integer.parseInt raises NumberFormatException. The inner catch constructs and throws InvalidQuantityException with the application message and that original object as its cause. The outer catch prints the message and Has cause: true. "-3" converts successfully, then fails the quantity < 0 guard. The new InvalidQuantityException has a null cause because there was no earlier conversion failure; the outer catch prints Has cause: false. The caller catches this specific application type for each input, so one failure does not discard the next input. The throws declaration states which checked type may leave read; it does not raise an exception on the successful path. NumberFormatException belongs to the RuntimeException family, so it is unchecked. InvalidQuantityException directly extends Exception and is outside the RuntimeException and Error families, so it is checked. A conventional Java method that lets that checked type escape must declare it or instead catch it within the method. These categories describe compiler rules; they do not decide whether a failure is harmless.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Quantity: 8
Problem: Quantity must be a whole number.
Has cause: true
Problem: Quantity cannot be negative.
Has cause: false
```

Common error: Treating successful conversion as proof that the input meets the application rule. Assuming every custom exception must have a cause. Treating a throws declaration as a statement that always raises an exception.

</details>

### Follow a checked failure through a helper

A quantity form uses `QuantityForm.read` to call the existing parser. The helper prints `Checking input` before the call and `Validated` only after a valid quantity returns. The outer caller reports the application message and, when present, the original cause message.

The first request supplies `"nine"`. Before running, predict the complete output, including the original conversion diagnostic. Locate the final handler. Identify the helper's return and the success messages that cannot be reached.

In [ ]:
Predicted complete output for "nine":

Final handler location:

Skipped return and success messages, with reasons:


In [ ]:
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
class QuantityForm {
    public static int read(String text) throws InvalidQuantityException {
        System.out.println("Checking input");
        int value = QuantityParser.read(text);
        System.out.println("Validated");
        return value;
    }
}
try {
    System.out.println("Value: " + QuantityForm.read("nine"));
} catch (InvalidQuantityException problem) {
    System.out.println("Problem: " + problem.getMessage());
    Throwable cause = problem.getCause();
    if (cause != null) {
        System.out.println("Cause: " + cause.getMessage());
    } else {
        System.out.println("Cause: none");
    }
}
System.out.println("After request");


Run the `"nine"` request and record the result. Distinguish the message written for the quantity task from the message supplied by the original conversion failure. Explain how the checked application exception passes through the helper and reaches the outer caller.

In [ ]:
Actual complete output:

Application message compared with the original cause message:

How the checked exception leaves the helper:

Comparison with my prediction:


The next complete program changes the form request to `"6"`. Its methods, handlers, and messages remain the same. Predict every output line before running. State whether the helper returns normally and whether the caller needs to inspect a cause on this path.

In [ ]:
Predicted complete output for "6":

Whether the helper returns and whether a cause is inspected:


In [ ]:
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
class QuantityForm {
    public static int read(String text) throws InvalidQuantityException {
        System.out.println("Checking input");
        int value = QuantityParser.read(text);
        System.out.println("Validated");
        return value;
    }
}
try {
    System.out.println("Value: " + QuantityForm.read("6"));
} catch (InvalidQuantityException problem) {
    System.out.println("Problem: " + problem.getMessage());
    Throwable cause = problem.getCause();
    if (cause != null) {
        System.out.println("Cause: " + cause.getMessage());
    } else {
        System.out.println("Cause: none");
    }
}
System.out.println("After request");


Run the `"6"` request and record the result. Compare its path with the failed `"nine"` request. Explain why the helper can return a value even though its method header declares a possible checked exception.

In [ ]:
Actual successful output:

Difference from the failed path:

Why throws does not raise a failure by itself:


Now change only the argument in the form call from `"6"` to `"-2"`. Leave the parser's nonnegative rule unchanged. Before running, predict all output and identify which cause-reporting branch the caller will select. Explain why this path differs from the `"nine"` request.

In [ ]:
Predicted complete output for "-2":

Predicted cause branch and why it differs from "nine":


Run the `"-2"` request and record the output. Explain why the caller checks for `null` before reading a cause message. What would go wrong if it tried to call `getMessage()` through a missing cause reference?

Restore the argument to `"6"`, rerun, and record the restored result. Distinguish the helper's checked-exception declaration from the compiler rules for the original unchecked `NumberFormatException`.

In [ ]:
Actual negative-input output:

Why the null check is needed:

What a method call through a null cause would do:

Actual restored output for "6":

Checked declaration compared with unchecked propagation:


<details>
<summary>Show answer</summary>

QuantityForm.read prints Checking input, then calls QuantityParser.read. With "nine", parsing raises NumberFormatException, and the parser throws a new InvalidQuantityException that retains it. The checked application exception leaves QuantityForm.read through its declared throws contract; Validated and that method's return are skipped. The outer catch receives the application exception. getCause returns its original parsing exception, so the non-null branch safely reads its message. In the selected Workspace runtime that message is For input string: "nine". After request prints after handling. With "6", both methods return normally: Validated and Value: 6 print even though the helper declares throws. With "-2", conversion succeeds but the nonnegative rule fails; the attached cause is null, so the caller prints Cause: none without calling getMessage through null. The helper could instead handle the checked type itself; in this program it declares that the type may continue outward to its caller.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
class QuantityForm {
    public static int read(String text) throws InvalidQuantityException {
        System.out.println("Checking input");
        int value = QuantityParser.read(text);
        System.out.println("Validated");
        return value;
    }
}
try {
    System.out.println("Value: " + QuantityForm.read("nine"));
} catch (InvalidQuantityException problem) {
    System.out.println("Problem: " + problem.getMessage());
    Throwable cause = problem.getCause();
    if (cause != null) {
        System.out.println("Cause: " + cause.getMessage());
    } else {
        System.out.println("Cause: none");
    }
}
System.out.println("After request");
```

Expected output:

```text
Checking input
Problem: Quantity must be a whole number.
Cause: For input string: "nine"
After request
```

Common error: Removing the helper declaration without adding its own handler in conventional Java source. Reading a cause message before checking whether the cause is null. Expecting Validated after the parser has thrown the application exception.

**Check case 2.** Both parser and helper return normally. Declaring a possible failure does not raise it; the caller prints the returned value.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
class QuantityForm {
    public static int read(String text) throws InvalidQuantityException {
        System.out.println("Checking input");
        int value = QuantityParser.read(text);
        System.out.println("Validated");
        return value;
    }
}
try {
    System.out.println("Value: " + QuantityForm.read("6"));
} catch (InvalidQuantityException problem) {
    System.out.println("Problem: " + problem.getMessage());
    Throwable cause = problem.getCause();
    if (cause != null) {
        System.out.println("Cause: " + cause.getMessage());
    } else {
        System.out.println("Cause: none");
    }
}
System.out.println("After request");
```

Expected output:

```text
Checking input
Validated
Value: 6
After request
```

**Check case 3.** Parsing succeeds, so the range failure has no earlier exception attached. The null check selects Cause: none and prevents a method call through null.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
class QuantityForm {
    public static int read(String text) throws InvalidQuantityException {
        System.out.println("Checking input");
        int value = QuantityParser.read(text);
        System.out.println("Validated");
        return value;
    }
}
try {
    System.out.println("Value: " + QuantityForm.read("-2"));
} catch (InvalidQuantityException problem) {
    System.out.println("Problem: " + problem.getMessage());
    Throwable cause = problem.getCause();
    if (cause != null) {
        System.out.println("Cause: " + cause.getMessage());
    } else {
        System.out.println("Cause: none");
    }
}
System.out.println("After request");
```

Expected output:

```text
Checking input
Problem: Quantity cannot be negative.
Cause: none
After request
```

</details>

### Complete a checked exception path

A training coordinator records each participant’s completed training level as an integer. Level zero is allowed; negative levels are not. A negative level must produce an `InvalidLevelException`, while a later valid level should still be checked. These inputs are already integers, so this example performs no text conversion.

The displayed draft is incomplete and for reading only. Choose `super`, `throws`, and `throw` for `CALL_PARENT`, `DECLARE_FAILURE`, and `RAISE_FAILURE`, using each once. Explain their separate jobs and predict the full output for `{-2, 1}`. Then copy the draft into the Java work cell and apply those replacements. Keep all other statements unchanged.

```java
class InvalidLevelException extends Exception {
    public InvalidLevelException(String message, Throwable cause) {
        CALL_PARENT(message, cause);
    }
}
class LevelRules {
    public static int requireNonnegative(int level) DECLARE_FAILURE InvalidLevelException {
        if (level < 0) {
            RAISE_FAILURE new InvalidLevelException("Level cannot be negative.", null);
        }
        return level;
    }
}
int[] levels = {-2, 1};
for (int level : levels) {
    try {
        System.out.println("Level: " + LevelRules.requireNonnegative(level));
    } catch (InvalidLevelException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

In [ ]:
CALL_PARENT replacement and its job:

DECLARE_FAILURE replacement and its job:

RAISE_FAILURE replacement and its job:

Predicted complete output:


Run your completed program and record every output line. Compare the result with your prediction. Explain why `InvalidLevelException` is checked, where the caller handles it, and why the rejected negative level has no earlier conversion cause.

In [ ]:
Actual complete output:

Prediction comparison:

Why the custom type is checked and where it is handled:

Why the negative level has no earlier cause:


<details>
<summary>Show answer</summary>

CALL_PARENT is super: the exception constructor passes its message and cause to the Exception constructor. DECLARE_FAILURE is throws: requireNonnegative states that InvalidLevelException may reach its caller. RAISE_FAILURE is throw: the negative branch raises a new object now. The input -2 takes that branch, so the caller prints the problem and Has cause: false. There is no earlier exception attached because this is a direct range check. Input 1 reaches return, so Level: 1 prints. The specific catch meets the caller's obligation for this checked type.

```java
class InvalidLevelException extends Exception {
    public InvalidLevelException(String message, Throwable cause) {
        super(message, cause);
    }
}
class LevelRules {
    public static int requireNonnegative(int level) throws InvalidLevelException {
        if (level < 0) {
            throw new InvalidLevelException("Level cannot be negative.", null);
        }
        return level;
    }
}
int[] levels = {-2, 1};
for (int level : levels) {
    try {
        System.out.println("Level: " + LevelRules.requireNonnegative(level));
    } catch (InvalidLevelException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Problem: Level cannot be negative.
Has cause: false
Level: 1
```

Common error: Swapping throw and throws even though one is a statement and the other is part of a method declaration. Dropping the cause argument from the supplied constructor call. Placing the success print outside the protected method call.

</details>

### Accept surrounding spaces before conversion

The supply assistant may enter spaces around a quantity. Recall that `text.trim()` returns a string with leading and trailing spaces removed; it does not change the original string in place. Passing that returned string to `Integer.parseInt` changes the text presented for conversion.

In the next program, replace only `Integer.parseInt(text)` with `Integer.parseInt(text.trim())`. Keep the exception class, nonnegative rule, handlers, messages, and initial inputs `{"8", "oops", "-3"}` unchanged. Predict the complete output before editing and running.

In [ ]:
Predicted complete output after adding trim:

Text supplied to parseInt for each initial input:


In [ ]:
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}


Run the edited program with the original inputs and record the result. Explain why removing surrounding spaces does not change the nonnegative rule or remove the requirement to preserve an actual parsing failure as a cause.

In [ ]:
Actual output for the original inputs:

Why the range rule is unchanged:

Why a parsing failure still needs its cause preserved:


Keep the trimming change and test two input arrays separately: `{" 8 ", "oops", "-3"}` and `{"   "}`. The second contains one string made only of spaces; it is not an empty array.

Before running either case, predict every output line. State what text reaches `parseInt` after trimming each changed input, and decide whether that text can represent an integer.

In [ ]:
Predicted output for {" 8 ", "oops", "-3"}:

Predicted output for {"   "}:

Text passed to parseInt after trimming the changed entries:

Whether each trimmed entry can represent an integer:


Run each variant and record its complete output. Compare the results with your predictions. Explain why removing surrounding spaces does not turn every input into valid integer text. Restore `{"8", "oops", "-3"}`, rerun, and record that result too.

In [ ]:
Actual output for {" 8 ", "oops", "-3"}:

Actual output for {"   "}:

Prediction comparison and the limit of trimming:

Actual restored baseline output:


<details>
<summary>Show answer</summary>

Trimming removes surrounding whitespace before conversion. The original inputs have no surrounding spaces, so this edit leaves their output unchanged. The padded " 8 " becomes "8" and converts successfully, while "oops" still fails conversion and "-3" still fails the nonnegative rule. A whitespace-only String becomes empty after trim, so Integer.parseInt still raises NumberFormatException and the custom exception retains it. The edit changes accepted text formatting; it does not change the application's numeric rule or the cause argument.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text.trim());
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Quantity: 8
Problem: Quantity must be a whole number.
Has cause: true
Problem: Quantity cannot be negative.
Has cause: false
```

Common error: Assuming trim converts words into numbers. Changing the range guard while editing input formatting. Removing the original cause when changing the parse call.

**Additional test: `Padded numeric input with other paths unchanged`.** Trimming the spaces makes the first input a valid 8. The nonnumeric and negative paths still preserve their original meanings and cause flags.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text.trim());
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {" 8 ", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Quantity: 8
Problem: Quantity must be a whole number.
Has cause: true
Problem: Quantity cannot be negative.
Has cause: false
```

**Additional test: `Whitespace-only input`.** After trimming, no digits remain. Conversion still fails and its original NumberFormatException remains attached to the custom exception.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text.trim());
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"   "};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Problem: Quantity must be a whole number.
Has cause: true
```

</details>

### Repair a report that loses the original cause

The supply assistant still receives an application message for invalid text, but a developer also needs the original conversion failure for diagnosis. The draft below loses that information: its conversion handler passes `null` to the custom exception constructor.

Keep this faulty draft in the reading cell. Predict its complete output and identify the report that violates the requirement to retain the original cause. Plan a repair to that one constructor argument and predict the repaired output before copying the complete program into the Java work cell. Preserve the messages, inputs, range check, caller, and the `null` argument used for a successfully parsed negative value.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", null);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

In [ ]:
Predicted faulty output:

Report that violates the cause requirement:

My one-argument repair and why:

Predicted repaired output:


Run the repaired program and record every output line. Explain why checking only the successful value and the application message would not reveal the lost cause. Why should the range-failure branch still use `null` for a successfully parsed negative value?

In [ ]:
Actual repaired output:

Why values and application messages alone miss this defect:

Why the negative-value branch still has no earlier cause:


<details>
<summary>Show answer</summary>

The text "8" converts to 8 and passes the nonnegative rule, so Quantity: 8 prints. "oops" cannot convert: Integer.parseInt raises NumberFormatException. The inner catch constructs and throws InvalidQuantityException with the application message and that original object as its cause. The outer catch prints the message and Has cause: true. "-3" converts successfully, then fails the quantity < 0 guard. The new InvalidQuantityException has a null cause because there was no earlier conversion failure; the outer catch prints Has cause: false. The caller catches this specific application type for each input, so one failure does not discard the next input. The throws declaration states which checked type may leave read; it does not raise an exception on the successful path. In the faulty conversion catch, passing null discards the original NumberFormatException even though the application message still describes the conversion failure. The repaired constructor call passes cause, the actual caught object. The direct negative-value rejection correctly keeps null because conversion succeeded. Checking the cause flag distinguishes the faulty and repaired programs; checking only the visible problem sentence would miss the defect.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Quantity: 8
Problem: Quantity must be a whole number.
Has cause: true
Problem: Quantity cannot be negative.
Has cause: false
```

Common error: Replacing the conversion message without restoring its original cause. Changing both null arguments even though the range failure has no earlier exception. Treating the cause flag as the success result of parsing.

</details>

## Independent Practice

### Build a seat-count parser with a clear failure contract

A registration assistant enters the number of seats requested for a booking. Each request arrives as text and is checked independently. A booking must request at least one seat. The program should accept a positive count or explain why that request was rejected, preserving an earlier conversion failure when one exists.

Create `InvalidSeatCountException extends Exception` with a constructor taking `String message` and `Throwable cause`; pass both arguments to `super`. Create `SeatParser` with `public static int read(String text)` declaring `throws InvalidSeatCountException`.

Parse with `Integer.parseInt(text)`. If conversion raises `NumberFormatException`, throw a new `InvalidSeatCountException` with the message `Seats must be a whole number.` and the original cause. Reject a parsed value below one with `Seats must be positive.` and a `null` cause. Otherwise, return the count.

Process `String[] inputs = {"4", "many", "0"};` with a specific `InvalidSeatCountException` handler inside each iteration. On success, print `Seats: ` plus the returned count. On failure, print `Problem: ` plus the message, then `Has cause: ` plus whether `getCause()` is non-null. Keep complete callers inside `try`/`catch`, or declare the checked exception from any helper method that lets it escape.

Before writing the complete exception class, parser, and caller in the Java work cell, outline their responsibilities and predict every output line.

In [ ]:
My exception class, parser rule, and caller responsibilities:

Predicted complete baseline output:


Run your program with `{"4", "many", "0"}` and record its complete output. Identify where the checked failure is declared or handled. Explain why zero is rejected for this booking even though the quantity parser accepted zero, and why the two rejected entries have different cause flags.

In [ ]:
Actual complete baseline output:

Where the checked failure is declared or handled:

Why zero differs from the quantity example:

Cause flag on each rejected path and why:


### Check the smallest valid value and conversion limits

Keep the complete `SeatParser` program unchanged while testing these arrays separately: `{"4", "many", "0"}`, `{"1"}`, `{"-2"}`, `{"2147483648"}`, and `{" 1 "}`. Change only the `inputs` initializer.

The largest Java `int` value is `2147483647`. Text can therefore contain digits and still fail integer conversion because its value is too large. This fixed seat parser uses `Integer.parseInt(text)` without the trimming change from guided practice.

Before running each case, predict every output line. Identify whether it succeeds, fails during conversion, or reaches the application rule and is rejected there.

In [ ]:
Predicted output and path for {"4", "many", "0"}:

Predicted output and path for {"1"}:

Predicted output and path for {"-2"}:

Predicted output and path for {"2147483648"}:

Predicted output and path for {" 1 "}:


Run each case and record the actual output. Explain which case checks the smallest valid seat count, which reaches the positive-count rule after successful conversion, and which cases fail before that rule is checked. Identify which rejected entries preserve an earlier exception.

If a result differs from the contract, repair the program and repeat the affected tests. Restore `{"4", "many", "0"}` and record one final run. Explain why useful tests must check the cause flag as well as the application message.

In [ ]:
Actual output for {"4", "many", "0"}:

Actual output for {"1"}:

Actual output for {"-2"}:

Actual output for {"2147483648"}:

Actual output for {" 1 "}:

What the lower boundary, negative value, oversized value, and spaces check:

Which cases reach the value rule and which retain an earlier cause:

Corrections and repeated checks, or why none were needed:

Actual restored baseline output:

Why both the message and cause flag need checking:


<details>
<summary>Show answer</summary>

The custom type directly extends Exception and is checked. Its one constructor preserves both message and cause. SeatParser.read declares that type, wraps NumberFormatException without losing the original object, and separately rejects counts below one. The text "4" returns 4. "many" fails conversion, so the application message is accompanied by Has cause: true. "0" converts successfully but fails the positive-count rule, so its cause is null and Has cause: false prints. Each iteration has a specific handler, allowing every input to receive its report. A conventional method that lets the checked type continue outward must declare it; otherwise it needs its own handler. The value 1 checks the smallest accepted seat count. The negative value -2 parses but fails the application rule with no earlier cause. The value 2147483648 is outside the int range and cannot be converted, so its original NumberFormatException is preserved. The surrounding spaces in " 1 " also prevent conversion in this unchanged parser. Its numeric guard is never reached on either conversion failure. The tests distinguish the two failure paths and would expose a parser that loses the original cause or incorrectly accepts zero.

```java
class InvalidSeatCountException extends Exception {
    public InvalidSeatCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class SeatParser {
    public static int read(String text) throws InvalidSeatCountException {
        int seats;
        try {
            seats = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidSeatCountException("Seats must be a whole number.", cause);
        }
        if (seats < 1) {
            throw new InvalidSeatCountException("Seats must be positive.", null);
        }
        return seats;
    }
}
String[] inputs = {"4", "many", "0"};
for (String input : inputs) {
    try {
        System.out.println("Seats: " + SeatParser.read(input));
    } catch (InvalidSeatCountException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Seats: 4
Problem: Seats must be a whole number.
Has cause: true
Problem: Seats must be positive.
Has cause: false
```

Common error: Reusing the quantity < 0 guard when seat counts must be at least one. Passing null when wrapping the original NumberFormatException. Attaching an invented parsing failure when conversion actually succeeded. Leaving a checked failure neither handled nor declared in conventional Java methods. Changing the exact method names, messages or caller inputs.

**Additional test: {"1"}.** One is the smallest valid seat count. Conversion succeeds and the positive-count rule accepts it, so there is no exception report.

```java
class InvalidSeatCountException extends Exception {
    public InvalidSeatCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class SeatParser {
    public static int read(String text) throws InvalidSeatCountException {
        int seats;
        try {
            seats = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidSeatCountException("Seats must be a whole number.", cause);
        }
        if (seats < 1) {
            throw new InvalidSeatCountException("Seats must be positive.", null);
        }
        return seats;
    }
}
String[] inputs = {"1"};
for (String input : inputs) {
    try {
        System.out.println("Seats: " + SeatParser.read(input));
    } catch (InvalidSeatCountException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Seats: 1
```

**Additional test: {"-2"}.** The text converts to a negative integer, then fails the positive-count rule. No earlier parsing exception exists, so the cause is null.

```java
class InvalidSeatCountException extends Exception {
    public InvalidSeatCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class SeatParser {
    public static int read(String text) throws InvalidSeatCountException {
        int seats;
        try {
            seats = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidSeatCountException("Seats must be a whole number.", cause);
        }
        if (seats < 1) {
            throw new InvalidSeatCountException("Seats must be positive.", null);
        }
        return seats;
    }
}
String[] inputs = {"-2"};
for (String input : inputs) {
    try {
        System.out.println("Seats: " + SeatParser.read(input));
    } catch (InvalidSeatCountException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Problem: Seats must be positive.
Has cause: false
```

**Additional test: {"2147483648"}.** These digits represent a value larger than the maximum int value, 2147483647. Integer.parseInt cannot produce an int and raises NumberFormatException. The application exception retains that cause; the range check is not reached.

```java
class InvalidSeatCountException extends Exception {
    public InvalidSeatCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class SeatParser {
    public static int read(String text) throws InvalidSeatCountException {
        int seats;
        try {
            seats = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidSeatCountException("Seats must be a whole number.", cause);
        }
        if (seats < 1) {
            throw new InvalidSeatCountException("Seats must be positive.", null);
        }
        return seats;
    }
}
String[] inputs = {"2147483648"};
for (String input : inputs) {
    try {
        System.out.println("Seats: " + SeatParser.read(input));
    } catch (InvalidSeatCountException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Problem: Seats must be a whole number.
Has cause: true
```

**Additional test: {" 1 "}.** The fixed SeatParser uses Integer.parseInt without trimming. Surrounding spaces prevent conversion, even though the central digit is positive. The original NumberFormatException is retained as the cause.

```java
class InvalidSeatCountException extends Exception {
    public InvalidSeatCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class SeatParser {
    public static int read(String text) throws InvalidSeatCountException {
        int seats;
        try {
            seats = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidSeatCountException("Seats must be a whole number.", cause);
        }
        if (seats < 1) {
            throw new InvalidSeatCountException("Seats must be positive.", null);
        }
        return seats;
    }
}
String[] inputs = {" 1 "};
for (String input : inputs) {
    try {
        System.out.println("Seats: " + SeatParser.read(input));
    } catch (InvalidSeatCountException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Problem: Seats must be a whole number.
Has cause: true
```

</details>

## Summary

An exception contract describes how a method reports failure as well as what it returns on success. An executed `throw` raises an exception object; `throws` declares a possible failure in a method header. Checked types require ordinary Java method code to catch or declare them.

Parsing and validation answer different questions. A string may fail to represent an integer, or a parsed integer may break the application rule. Preserve an earlier conversion exception as a cause when one exists, and give the caller a useful application message.

### Retrieve the main idea

Without reopening the worked example, explain how `"-1"` can pass integer conversion but fail the quantity rule. Distinguish the application message from an attached cause. Then explain the separate jobs of an executed `throw` statement and a method's `throws` declaration.

In [ ]:
Why conversion can succeed while validation fails:

Application message compared with an attached cause:

Executed throw compared with a throws declaration:


<details>
<summary>Show answer</summary>

The text `"-1"` represents an integer, so conversion succeeds. The quantity rule then rejects that negative value. The custom exception carries the application message but has no earlier conversion failure to attach.

A message describes this failure; a cause links an earlier exception object. Executing `throw` raises the particular object and interrupts normal completion. A `throws` declaration tells callers that a checked failure may escape; it does not raise anything by itself. Valid calls can still return normally.
</details>

## Reflection

Choose a seat-count rule for another registration system and state it precisely. Describe one text entry that fails conversion and one successfully parsed integer that violates your chosen rule. For each failure, explain what message the application should provide and whether there is an earlier exception to preserve as a cause.

In [ ]:
My registration rule:

Conversion-failure example, application message, and earlier cause:

Value-rule failure example, application message, and cause choice:


### Looking Ahead

You can now report invalid input without losing the reason for a failure. Some methods also hold resources, such as an open file. The next lesson explains how cleanup can run when work succeeds or fails.

## Supplemental Reading

- [Throwing Java exceptions](https://dev.java/learn/exceptions/throwing/) covers throw, method declarations, and custom types.
- [Java 21 exception checking](https://docs.oracle.com/javase/specs/jls/se21/html/jls-11.html) states the checked and unchecked rules.
- [Java 21 Exception API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Exception.html) documents constructors that retain a message and cause.
- [Java 21 Integer.parseInt](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Integer.html#parseInt(java.lang.String)) specifies decimal conversion and failure cases.
